In [1]:
!pip install geopandas
!pip install gdown
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry 
import Point
import contextily as ctx
import gdown
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal
import seasonal_decompose
from numpy 
import radians
from windrose 
import WindroseAxes
import folium
from folium.plugins 
import HeatMap
from sklearn.preprocessing 
import MinMaxScaler
from sklearn.model_selection 
import train_test_split
from sklearn.metrics 
import mean_absolute_error, mean_squared_error
from tensorflow.keras.models 
import Sequential
from tensorflow.keras.layers 
import Dense, LSTM, Dropout
import contextily as ctx
from sklearn.metrics 
import r2_score
import WindroseAxes

ModuleNotFoundError: No module named 'windrose'

In [23]:
# Load the dataset into a Pandas DataFrame
# Google Drive file ID (extracted from your link)
file_id = "1aQmuS8mzgYMDW4vNRIW4uQzpO-GPIMnF"
download_url = f"https://drive.google.com/uc?id={file_id}"

# Download the file
output_file = "hrly_Irish_weather.csv"
gdown.download(download_url, output_file, quiet=False)

# Load CSV into Pandas
df = pd.read_csv(output_file)

# Display the first few rows of the dataset
print(df.head())

Downloading...
From (original): https://drive.google.com/uc?id=1aQmuS8mzgYMDW4vNRIW4uQzpO-GPIMnF
From (redirected): https://drive.google.com/uc?id=1aQmuS8mzgYMDW4vNRIW4uQzpO-GPIMnF&confirm=t&uuid=1860ff0e-f690-4f8c-9c6e-4e834c1ec658
To: C:\Users\banch\CCT\Feb CA1\msc-feb2024-capstone\hrly_Irish_weather.csv
100%|███████████████████████████████████████████████████████████████████████████████| 486M/486M [00:06<00:00, 78.4MB/s]
C:\Users\banch\AppData\Local\Temp\ipykernel_30684\4260615717.py:11: DtypeWarning: Columns (5,6,7,8,9,10,11,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_file)


   county  station  latitude  longitude               date rain  temp  wetb  \
0  Galway  ATHENRY    53.289     -8.786  26-jun-2011 01:00  0.0  15.3  14.5   
1  Galway  ATHENRY    53.289     -8.786  26-jun-2011 02:00  0.0  14.7  13.7   
2  Galway  ATHENRY    53.289     -8.786  26-jun-2011 03:00  0.0  14.3  13.4   
3  Galway  ATHENRY    53.289     -8.786  26-jun-2011 04:00  0.0  14.4  13.6   
4  Galway  ATHENRY    53.289     -8.786  26-jun-2011 05:00  0.0  14.4  13.5   

  dewpt vappr rhum     msl wdsp wddir  sun  vis clht clamt  
0  13.9  15.8   90  1016.0    8   190  NaN  NaN  NaN   NaN  
1  12.9  14.9   89  1015.8    7   190  NaN  NaN  NaN   NaN  
2  12.6  14.6   89  1015.5    6   190  NaN  NaN  NaN   NaN  
3  12.8  14.8   90  1015.3    7   180  NaN  NaN  NaN   NaN  
4  12.7  14.7   89  1015.1    6   190  NaN  NaN  NaN   NaN  


In [25]:
# Display basic information about the dataset
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4660423 entries, 0 to 4660422
Data columns (total 18 columns):
 #   Column     Dtype  
---  ------     -----  
 0   county     object 
 1   station    object 
 2   latitude   float64
 3   longitude  float64
 4   date       object 
 5   rain       object 
 6   temp       object 
 7   wetb       object 
 8   dewpt      object 
 9   vappr      object 
 10  rhum       object 
 11  msl        object 
 12  wdsp       object 
 13  wddir      object 
 14  sun        object 
 15  vis        object 
 16  clht       object 
 17  clamt      object 
dtypes: float64(2), object(16)
memory usage: 640.0+ MB
None


In [27]:
# Display summary statistics
print(df.describe())


           latitude     longitude
count  4.660423e+06  4.660423e+06
mean   5.325453e+01 -8.181232e+00
std    9.898850e-01  1.220681e+00
min    5.147600e+01 -1.024100e+01
25%    5.229800e+01 -8.993000e+00
50%    5.342800e+01 -8.244000e+00
75%    5.390600e+01 -7.310000e+00
max    5.537200e+01 -6.241000e+00


In [29]:
# Check for missing values
print(df.isnull().sum())

county             0
station            0
latitude           0
longitude          0
date               0
rain               0
temp               0
wetb               0
dewpt              0
vappr              0
rhum               0
msl                0
wdsp          229032
wddir         229032
sun          2585167
vis          2585167
clht         2585167
clamt        2585167
dtype: int64


In [31]:
# Drop irrelevant columns
df.drop(columns=['sun', 'vis', 'clht', 'clamt'], inplace=True)

# Clean and convert 'wdsp' column
df['wdsp'].replace(' ', np.nan, inplace=True)  # Replace empty spaces with NaN
df['wdsp'] = pd.to_numeric(df['wdsp'], errors='coerce')  # Convert to numeric
df['wdsp'].fillna(df['wdsp'].mean(), inplace=True)  # Fill missing values with mean

# Clean and convert 'wddir' column
df['wddir'].replace(' ', np.nan, inplace=True)  # Replace empty spaces with NaN
df['wddir'] = pd.to_numeric(df['wddir'], errors='coerce')  # Convert to numeric
df['wddir'].fillna(df['wddir'].mean(), inplace=True)  # Fill missing values with mean

# Ensure 'temp' column is numeric
df['temp'] = pd.to_numeric(df['temp'], errors='coerce')  # Convert to numeric


C:\Users\banch\AppData\Local\Temp\ipykernel_30684\1061589631.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['wdsp'].replace(' ', np.nan, inplace=True)  # Replace empty spaces with NaN
C:\Users\banch\AppData\Local\Temp\ipykernel_30684\1061589631.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

In [33]:
# Check for missing values after handling
print("Missing values after handling:")
print(df.isnull().sum())

Missing values after handling:
county           0
station          0
latitude         0
longitude        0
date             0
rain             0
temp         32581
wetb             0
dewpt            0
vappr            0
rhum             0
msl              0
wdsp             0
wddir            0
dtype: int64


In [35]:
print(df.describe(include='all'))

         county    station      latitude     longitude               date  \
count   4660423    4660423  4.660423e+06  4.660423e+06            4660423   
unique       15         25           NaN           NaN             266617   
top        Mayo  BELMULLET           NaN           NaN  26-jun-2011 01:00   
freq     877279     266617           NaN           NaN                 25   
mean        NaN        NaN  5.325453e+01 -8.181232e+00                NaN   
std         NaN        NaN  9.898850e-01  1.220681e+00                NaN   
min         NaN        NaN  5.147600e+01 -1.024100e+01                NaN   
25%         NaN        NaN  5.229800e+01 -8.993000e+00                NaN   
50%         NaN        NaN  5.342800e+01 -8.244000e+00                NaN   
75%         NaN        NaN  5.390600e+01 -7.310000e+00                NaN   
max         NaN        NaN  5.537200e+01 -6.241000e+00                NaN   

           rain          temp     wetb    dewpt    vappr     rhum      msl 

In [ ]:
print("Shape of the dataset:", df.shape)
print("Columns in the dataset:", df.columns)

In [ ]:
print(df[['rain', 'temp', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl', 'wdsp']].dtypes)
print(df[['rain', 'temp', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl', 'wdsp']].apply(lambda x: pd.to_numeric(x, errors='coerce')).isnull().sum())


In [ ]:
cols_to_clean = ['rain', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl']
for col in cols_to_clean:
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
df[col] = df[col].fillna(df[col].mean())

In [ ]:
print(df[cols_to_clean].dtypes)
print(df[cols_to_clean].isnull().sum())


In [ ]:
# Counties and Stations
df['county'].value_counts().plot(kind='bar', figsize=(10, 5), title="Data Count per County")
plt.show()

In [ ]:
# Correlation
corr_matrix = df[['rain', 'temp', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl', 'wdsp']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Between Weather Features")
plt.show()

In [ ]:
# Convert longitude and latitude to geometry points
geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
geo_df = gpd.GeoDataFrame(df, geometry=geometry)

# Set coordinate reference system (CRS) to WGS84 (EPSG:4326)
geo_df.set_crs(epsg=4326, inplace=True)

# Convert to Web Mercator CRS (EPSG:3857) for compatibility with basemaps
geo_df = geo_df.to_crs(epsg=3857)

# Plot on a basemap
fig, ax = plt.subplots(figsize=(10, 10))
geo_df.plot(ax=ax, markersize=10, color='blue', alpha=0.5, label='Stations')

# Add a basemap using OpenStreetMap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Weather Stations in Ireland")
plt.legend()
plt.show()

In [ ]:
# Outliers Detection
sns.boxplot(df['rain']).set_title("Rainfall Outliers")
plt.show()

In [ ]:
# Plot temperature over time
plt.figure(figsize=(12, 6))
df['temp'].plot(label="Temperature")
plt.title("Temperature Trends Over Time")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.grid()
plt.show()

In [ ]:
print(df['date'].isnull().sum())  
df = df.dropna(subset=['date'])  

In [ ]:
df = df[df.index.notnull()]
df['date'] = pd.to_datetime(df['date'], errors='coerce')  
df.set_index('date', inplace=True)  

In [ ]:
plt.figure(figsize=(12, 6))
df['temp'].resample('M').mean().plot(label='Average Temperature', color='red')
df['rain'].resample('M').mean().plot(label='Average Rainfall', color='blue')

plt.title("Rainfall and Temperature Trends Over Time")
plt.xlabel("Date")
plt.ylabel("Values")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Wind Speed Vs. Wind Direction
plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
ax.scatter(radians(df['wddir']), df['wdsp'], alpha=0.6, c=df['wdsp'], cmap='viridis')
ax.set_title("Wind Speed vs Wind Direction (Polar Plot)", va='bottom')
plt.show()

In [ ]:
from windrose import WindroseAxes
# Wind Rose Plot
ax = WindroseAxes.from_ax()
ax.bar(df['wddir'], df['wdsp'], normed=True, opening=0.8, edgecolor='white')
ax.set_legend()
plt.title("Wind Rose Plot")
plt.show()

In [ ]:
# Histogram of Temperature
plt.figure(figsize=(10, 6))
df['temp'].plot(kind='hist', bins=50, alpha=0.7, color='orange')
plt.title("Histogram of Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.grid()
plt.show()

In [ ]:
# Average Rainfall by County
county_rain = df.groupby('county')['rain'].mean()  # Average rainfall by county
county_rain = county_rain.sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=county_rain.index, y=county_rain.values, palette='Blues_d')
plt.title("Average Rainfall by County")
plt.xlabel("County")
plt.ylabel("Average Rainfall")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#Hourly Trends of Temperature and Rainfall
df['hour'] = df.index.hour  # Extract the hour from the datetime index
hourly_temp = df.groupby('hour')['temp'].mean()
hourly_rain = df.groupby('hour')['rain'].mean()

plt.figure(figsize=(12, 6))
plt.plot(hourly_temp, label="Temperature (°C)", color="red")
plt.plot(hourly_rain, label="Rainfall (mm)", color="blue")
plt.title("Hourly Trends of Temperature and Rainfall")
plt.xlabel("Hour")
plt.ylabel("Values")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Seasonal Trends
temp_decomposed = seasonal_decompose(df['temp'].resample('D').mean(), model='additive', period=365)
temp_decomposed.plot()
plt.show()

In [ ]:
# Scatter Plot: Relation Between Rainfall and Humidity
df = df[~df.index.duplicated()]  
df['rhum'] = pd.to_numeric(df['rhum'], errors='coerce')  
df['rain'] = pd.to_numeric(df['rain'], errors='coerce')  
df = df.dropna(subset=['rhum', 'rain'])  

plt.figure(figsize=(10, 6))
sns.scatterplot(x=df['rhum'], y=df['rain'], alpha=0.5, color='purple')
plt.title("Relationship Between Rainfall and Humidity")
plt.xlabel("Relative Humidity (%)")
plt.ylabel("Rainfall (mm)")
plt.grid()
plt.show()


In [ ]:
# Top Rainfall Days
top_rainfall = df.resample('D')['rain'].sum().nlargest(10)
plt.figure(figsize=(10, 6))
top_rainfall.plot(kind='bar', color='teal', alpha=0.7)
plt.title("Top Rainfall Days")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid()
plt.show()

In [ ]:
# Rainfall Intensity Heatmap
pivot_table = df.pivot_table(index=df.index.hour, columns=df.index.date, values='rain', aggfunc='mean')
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table, cmap='Blues', cbar_kws={'label': 'Rainfall (mm)'})
plt.title("Hourly Rainfall Intensity Heatmap")
plt.xlabel("Date")
plt.ylabel("Hour of Day")
plt.show()

In [ ]:
# Temperature vs Dew Point (Joint Distribution)
sns.jointplot(x='temp', y='dewpt', data=df, kind='hex', cmap='coolwarm', height=8)
plt.title("Temperature vs Dew Point (Joint Distribution)", loc='center')
plt.show()

In [ ]:
# Temperature Variance Over the Years
df['year'] = df.index.year
temp_variance = df.groupby('year')['temp'].var()
plt.figure(figsize=(10, 6))
temp_variance.plot(kind='line', color='green', marker='o')
plt.title("Yearly Temperature Variance")
plt.xlabel("Year")
plt.ylabel("Variance")
plt.grid()
plt.show()

In [ ]:
# Boxenplot of Weather Features by County
plt.figure(figsize=(12, 8))
sns.boxenplot(x='county', y='rain', data=df, palette='coolwarm')
plt.title("Rainfall Distribution by County")
plt.xlabel("County")
plt.ylabel("Rainfall (mm)")
plt.xticks(rotation=45)
plt.show()

In [ ]:

# Select relevant features for rainfall prediction
features = ['latitude', 'longitude', 'temp', 'rhum', 'dewpt', 'wdsp', 'wddir', 'msl', 'rain']
df_rain = df[features].dropna()

# Normalize data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_rain)

# Create sequences for time-series prediction
def create_sequences(data, sequence_length):
    x, y = [], []
    for i in range(len(data) - sequence_length):
        x.append(data[i:i + sequence_length, :-1])
        y.append(data[i + sequence_length, -1])
    return np.array(x), np.array(y)
# Use past 24 hours of data to predict the next rainfall value
sequence_length = 24
x, y = create_sequences(scaled_data, sequence_length)

# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Build LSTM model for rainfall forecasting
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], x_train.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)  # Predicting a single rainfall value
])

model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(x_train, y_train, validation_split=0.2, epochs=10, batch_size=64)

# Evaluate the model
y_pred = model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"R-squared: {r2:.2f}")


In [ ]:

# Forecast rainfall for specific locations
locations = df_rain[['latitude', 'longitude']].drop_duplicates()
predictions = []
for loc in locations.itertuples():
    loc_data = df_rain[(df_rain['latitude'] == loc.latitude) & (df_rain['longitude'] == loc.longitude)]
    loc_scaled = scaler.transform(loc_data)
    loc_x, loc_y = create_sequences(loc_scaled, sequence_length)
    loc_prediction = model.predict(loc_x)
    mean_rainfall = scaler.inverse_transform(
        np.concatenate((np.zeros((len(loc_prediction), scaled_data.shape[1] - 1)), loc_prediction), axis=1)
    )[:, -1].mean()
    predictions.append({
        "latitude": loc.latitude,
        "longitude": loc.longitude,
        "predicted_rainfall": mean_rainfall
    })

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Plot predictions on a real-world map
geo_df = gpd.GeoDataFrame(
    predictions_df, geometry=gpd.points_from_xy(predictions_df.longitude, predictions_df.latitude)
)
geo_df.set_crs(epsg=4326, inplace=True)
geo_df = geo_df.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 12))
geo_df.plot(
    ax=ax, markersize=geo_df['predicted_rainfall'] * 10,
    color='blue', alpha=0.5, label='Predicted Rainfall (mm)'
)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
plt.title("Predicted Rainfall in Specific Locations")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()


In [ ]:
# Rainfall heatmap
map = folium.Map(location=[geo_df.geometry.y.mean(), geo_df.geometry.x.mean()], zoom_start=7)
heat_data = [[row.geometry.y, row.geometry.x, row['predicted_rainfall']] for index, row in geo_df.iterrows()]
HeatMap(heat_data).add_to(map)
map.save("rainfall_heatmap.html")